# Notebook 1 – Prepare a Core Ortholog Group from OMA

fDOG-Assembly needs a **core ortholog group** to define the gene it is searching for.
A core group consists of:
- a FASTA file with protein sequences from multiple species
- a multiple-sequence alignment (MSA) derived from those sequences
- an HMM profile built from the MSA

All three files are created automatically by `fdog.addCoreGroup` once you provide
the input FASTA, which we created here. 

This notebook shows how to download an **OMA Group** from the
[OMA browser](https://omabrowser.org) and prepare it as a core group for fDOG-Assembly.

---

### What is an OMA Group?

OMA Groups (also called *OMA pairwise groups*) are sets of proteins that are
**all pairwise orthologs** of each other – i.e. every pair of sequences in the
group is a bidirectional best hit across species. Each group has a unique integer
ID and typically contains one sequence per species.

---

### Prerequisites
| Requirement | How to obtain |
|---|---|
| fDOG installed | `pip install fdog` |
| Python packages | `pip install requests biopython` |
| MUSCLE (≥ v5) or MAFFT | system package manager |
| HMMER (`hmmbuild`) | system package manager |

## 1 – Configuration

Please adapt this part to download the OMA group of your choice. 

In [1]:
import subprocess
from pathlib import Path

import requests

# ── Gene of interest ─────────────────────────────────────────────────────────
# Provide any UniProt accession for the seed protein.
# fDOG-Assembly will look for orthologs of THIS gene in the target assembly.
GENE_NAME    = "GAPDH"   # name used for the core-group folder (choose freely)
SEED_UNIPROT = "P04406"  # UniProt accession of the human seed protein (GAPDH)

# Version tag embedded in the FASTA headers (choose any short string).
# This same tag is used later as the --refSpec version in fdog.assembly.
OMA_VERSION  = "OMA2024"

# ── Output ───────────────────────────────────────────────────────────────────
# All example data lives inside examples/data/
DATA_DIR       = Path("data")
CORE_GROUP_DIR = DATA_DIR / "core_orthologs"
CORE_GROUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Gene name    : {GENE_NAME}")
print(f"Seed protein : {SEED_UNIPROT}")
print(f"Version tag  : {OMA_VERSION}")
print(f"Output dir   : {CORE_GROUP_DIR}")

Gene name    : GAPDH
Seed protein : P04406
Version tag  : OMA2024
Output dir   : data/core_orthologs


## 2 – Find the OMA Group for the Seed Protein

The OMA REST API lets us look up a protein by its UniProt accession and
retrieve the integer ID of the OMA Group it belongs to.

In [4]:
OMA_BASE = "https://omabrowser.org/api"

# Look up the protein entry in OMA
print(f"Fetching OMA entry for UniProt ID: {SEED_UNIPROT} ...")
resp = requests.get(
    f"{OMA_BASE}/protein/{SEED_UNIPROT}/",
    headers={"Accept": "application/json"},
    timeout=30,
)
resp.raise_for_status()
protein = resp.json()

seed_omaid   = protein["omaid"]
oma_group_id = protein.get("oma_group")  # integer group ID; 0 if the protein has no group

print(f"  OMA protein ID : {seed_omaid}")
print(f"  Species        : {protein['species']['species']}")
print(f"  OMA Group ID   : {oma_group_id}")

if not oma_group_id:
    raise ValueError(
        f"{SEED_UNIPROT} does not belong to an OMA Group (oma_group = 0). "
        "Try a different seed protein or use the HOG-based approach."
    )

Fetching OMA entry for UniProt ID: P04406 ...
  OMA protein ID : HUMAN10651
  Species        : Homo sapiens
  OMA Group ID   : 1471466


## 3 – Download Members of the OMA Group

The `/api/group/{group_id}/members/` endpoint returns proteins in the group,
including their sequences, species information, and canonical accessions.

In [5]:
import random
print(f"Fetching members of OMA Group {oma_group_id} ...")
resp_m = requests.get(
    f"{OMA_BASE}/group/{oma_group_id}/",
    headers={"Accept": "application/json"},
    timeout=60,
)
resp_m.raise_for_status()
group_data = resp_m.json()

if isinstance(group_data, dict):
    members = group_data.get("members", [])
else:
    members = group_data

print(f"Group contains {len(members)} sequences:")

## This group is very large, to make things easier we will contain only some selected species: Homo sapiens (refernce later), mouse, chicken, zebrafish and X.tropicalis
## please adapt this part to your needs
TARGET_SPECIES = {"HUMAN", "MOUSE", "CHICK", "DANRE", "XENTR"}

members = [
    m for m in members
    if m.get("omaid", "")[:5] in TARGET_SPECIES
    and m.get("species", {}).get("taxon_id", -1) > 0
]

print(f"Filtered to {len(members)} members:")
for m in members:
    print(f"  {m.get('omaid','?')[:5]}  taxon_id={m.get('species',{}).get('taxon_id','?')}")

print(f"Group contains {len(members)} sequences:")

for m in members:
    omaid = m.get("omaid")
    resp_p = requests.get(
        f"{OMA_BASE}/protein/{omaid}/",
        headers={"Accept": "application/json"},
        timeout=30,
    )
    resp_p.raise_for_status()
    protein_data = resp_p.json()
    m["sequence"] = protein_data.get("sequence", "")
    print(f"  {omaid}  {len(m['sequence'])} aa")

print(f"  {'OMA ID':<14} {'Species':<35} {'Canonical ID'}")
print(f"  {'-'*14} {'-'*35} {'-'*15}")
for m in members:
    sp   = m.get("species", {})
    code = sp.get("code", "?????")
    name = sp.get("species", "?")[:34]
    cid  = m.get("canonicalid", "?")
    print(f"  {m.get('omaid','?'):<14} {name:<35} {cid}")
    print(m.get("species", {}))


Fetching members of OMA Group 1471466 ...
Group contains 1710 sequences:
Filtered to 5 members:
  DANRE  taxon_id=7955
  XENTR  taxon_id=8364
  CHICK  taxon_id=9031
  HUMAN  taxon_id=9606
  MOUSE  taxon_id=10090
Group contains 5 sequences:
  DANRE10148  333 aa
  XENTR29908  334 aa
  CHICK03943  333 aa
  HUMAN10651  335 aa
  MOUSE48616  333 aa
  OMA ID         Species                             Canonical ID
  -------------- ----------------------------------- ---------------
  DANRE10148     Danio rerio                         G3P_DANRE
{'code': 'DANRE', 'taxon_id': 7955, 'species': 'Danio rerio', 'genome_url': 'https://omabrowser.org/api/genome/DANRE/'}
  XENTR29908     Xenopus tropicalis                  Q28HJ8
{'code': 'XENTR', 'taxon_id': 8364, 'species': 'Xenopus tropicalis', 'genome_url': 'https://omabrowser.org/api/genome/XENTR/'}
  CHICK03943     Gallus gallus                       G3P_CHICK
{'code': 'CHICK', 'taxon_id': 9031, 'species': 'Gallus gallus', 'genome_url': 'https://

## 4 – Format the FASTA File

fDOG-Assembly requires the core group FASTA headers to follow this format:

```
>GENENAME|ABBR@NCBI_TAXID@VERSION|PROTEIN_ID
```

where:
- `GENENAME` is the name of the gene/group (e.g. `GAPDH`), the name is up to you and should be meaningful
- `ABBR` is a short species abbreviation (OMA uses 5-letter codes that are
  compatible with fDOG, e.g. `HUMAN`)
- `NCBI_TAXID` is the NCBI taxonomy ID
- `VERSION` is any short version string (we use `OMA2024`)
- `PROTEIN_ID` is the canonical protein accession (e.g. `P04406`)

**Example:**
```
>GAPDH|HUMAN@9606@OMA2024|P04406
MVKVGVNGFGRIGRLVTRAAFNSGKVD...
```

In [6]:
# Record the reference species – used later as --refSpec in fdog.assembly
# We use the seed protein's species (human) as the reference. The seed was defined in the first cell
ref_species_entry = None

fasta_lines = []
for m in members:
    sp       = m.get("species", {})
    abbr     = sp.get("code", "UNKN")        # OMA 5-letter code (e.g. HUMAN)
    taxid    = sp.get("taxon_id", 0)         # NCBI taxonomy ID
    prot_id  = m.get("canonicalid")  # UniProt canonical ID
    sequence = m.get("sequence", "")

    if not sequence:
        print(f"  SKIP {abbr}: no sequence available")
        continue

    # Build the fDOG-compatible header
    species_tag = f"{abbr}@{taxid}@{OMA_VERSION}"
    header = f">{GENE_NAME}|{species_tag}|{prot_id}"

    fasta_lines.append(header)
    fasta_lines.append(sequence)

    # Remember the human entry as the reference species for later
    if m.get("omaid", "") == seed_omaid:
        ref_species_entry = species_tag

print(f"Formatted {len(fasta_lines)//2} sequences.")
print()
print("First two headers:")
for line in fasta_lines[:4]:
    if line.startswith(">"):
        print(f"  {line}")

Formatted 5 sequences.

First two headers:
  >GAPDH|DANRE@7955@OMA2024|G3P_DANRE
  >GAPDH|XENTR@8364@OMA2024|Q28HJ8


## 5 – Write the Input FASTA File

In [7]:
if len(fasta_lines) < 6:  # at least 3 sequences (header + seq each)
    raise ValueError(
        f"Only {len(fasta_lines)//2} sequences available – need at least 3 "
        "for a meaningful MSA and HMM."
    )

raw_fasta = DATA_DIR / f"{GENE_NAME}_oma_group.fa"
raw_fasta.write_text("\n".join(fasta_lines) + "\n")

print(f"Written to: {raw_fasta}")
print(f"  Sequences : {len(fasta_lines)//2}")
print(f"  File size : {raw_fasta.stat().st_size:,} bytes")

if ref_species_entry:
    print(f"\nReference species for fdog.assembly:")
    print(f"  --refSpec {ref_species_entry}")

Written to: data/GAPDH_oma_group.fa
  Sequences : 5
  File size : 1,851 bytes

Reference species for fdog.assembly:
  --refSpec HUMAN@9606@OMA2024


## 6 – Build the Core Group with `fdog.addCoreGroup`

`fdog.addCoreGroup` takes the input FASTA and:
1. Converts it to single-line format (one sequence per line)
2. Builds a multiple-sequence alignment with MUSCLE
3. Builds an HMM profile from the alignment with `hmmbuild`

The result is a folder `GENE_NAME/` inside `CORE_GROUP_DIR` containing
`GENE_NAME.fa`, `GENE_NAME.aln`, and `hmm_dir/GENE_NAME.hmm`.

In [8]:
cmd = [
    "fdog.addCoreGroup",
    "--fasta",    str(raw_fasta),
    "--out",      str(CORE_GROUP_DIR),
    "--geneName", GENE_NAME,
]

print("Running:", " ".join(cmd))
print()
### command out the part below if you want to run fDA outside this jupyter notebook
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"fdog.addCoreGroup failed (exit {result.returncode})")

Running: fdog.addCoreGroup --fasta data/GAPDH_oma_group.fa --out data/core_orthologs --geneName GAPDH

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             data/core_orthologs/GAPDH/GAPDH.aln
# output HMM file:                  data/core_orthologs/GAPDH/hmm_dir/GAPDH.hmm
# input alignment is asserted as:   protein
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     GAPDH                    5   335   333     0.42  0.590 

# CPU time: 0.25u 0.00s 00:00:00.25 Elapsed: 00:00:00.26
Core group located at data/core_orthologs/GAPDH/. Fasta fil

## 7 – Verify the Output

In [9]:
gene_dir = CORE_GROUP_DIR / GENE_NAME
expected = {
    "FASTA"  : gene_dir / f"{GENE_NAME}.fa",
    "MSA"    : gene_dir / f"{GENE_NAME}.aln",
    "HMM"    : gene_dir / "hmm_dir" / f"{GENE_NAME}.hmm",
}

all_ok = True
for label, path in expected.items():
    ok = path.exists() and path.stat().st_size > 0
    status = "OK" if ok else "MISSING"
    print(f"  [{status}] {label:6}  {path}")
    all_ok &= ok

print()
if all_ok:
    print("Core group created successfully!")
    print()
    print("Use these values in fdog.assembly (Notebook 3):")
    print(f"  --gene          {GENE_NAME}")
    print(f"  --coregroupPath {CORE_GROUP_DIR}")
    if ref_species_entry:
        print(f"  --refSpec       {ref_species_entry}")
else:
    print("Some files are missing. Check the fdog.addCoreGroup output above.")

  [OK] FASTA   data/core_orthologs/GAPDH/GAPDH.fa
  [OK] MSA     data/core_orthologs/GAPDH/GAPDH.aln
  [OK] HMM     data/core_orthologs/GAPDH/hmm_dir/GAPDH.hmm

Core group created successfully!

Use these values in fdog.assembly (Notebook 3):
  --gene          GAPDH
  --coregroupPath data/core_orthologs
  --refSpec       HUMAN@9606@OMA2024


## Summary

| Step | What happened |
|---|---|
| 2 | Found OMA Group ID for the seed protein via the OMA REST API |
| 3 | Downloaded all group member sequences, filtered for species of interest |
| 4 | Formatted headers as `>GENENAME\|ABBR@TAXID@OMA2024\|PROT_ID` |
| 5 | Saved the raw FASTA to `data/GAPDH_oma_group.fa` |
| 6 | Built the core group (MSA + HMM) with `fdog.addCoreGroup` |

**Next step:** `02_prepare_assembly_from_NCBI.ipynb` – download and prepare a
target genome assembly, which is the genome assembly we want to search for orthologs in.